# **Spark Tutorial**

Brief tutorial on Session Initialization and tools (ports) we'll use.

## **Spark Session**

We can now create the Spark session. 

With the following command we are asking the `master` (and the resource manager) to create an __application__ with the required resources and configurations. 

In this case, we are using all the default options (e.g. number of cores and number of executors), but we can also specify them by hand with `.config("spark.some.config", "value")`. 

The list of available configurations can be found [here](https://spark.apache.org/docs/latest/configuration.html).

In [1]:
from pyspark.sql import SparkSession
import socket

# This creates the DRIVER (your notebook) and connects it to:
# Spark Master running on:
#   spark://10.67.22.135:7077
#
# IMPORTANT:
# - 7077 = Spark internal cluster communication (NOT a web page)
# - Master schedules tasks to workers automatically

spark = SparkSession.builder \
    .master("spark://10.67.22.135:7077") \
    .appName("PySpark_Tutorial_Notebook") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/03 15:45:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


From the Spark Session we can access the Spark Context.

The Spark Context is the driver application program we will use to submit applications to Spark, and it is used to work with RDDs

In [2]:
sc = spark.sparkContext

sc

<SparkContext master=spark://10.67.22.135:7077 appName=PySpark_Tutorial_Notebook>

Some basic info of the Spark Session

In [4]:
print("App Name      :", sc.appName)
print("Master        :", sc.master)
print("App ID        :", sc.applicationId)

App Name      : PySpark_Tutorial_Notebook
Master        : spark://10.67.22.135:7077
App ID        : app-20260703154504-0001


Network sanity check: this checks if your notebook can physically reach the Spark master

In [9]:
try:
    socket.create_connection(("10.67.22.135", 7077), timeout=3)
    print("Network check: Spark master reachable ✔")
except Exception as e:
    print("Network check FAILED ✘")
    print(e)

Network check: Spark master reachable ✔


## **Spark Ports Used in This Setup**

##### 7077 → Spark Master (Cluster Communication)
- Used by `SparkContext` / `SparkSession`
- Handles communication between driver, master, and worker nodes
- Used for job scheduling and cluster coordination
- Not accessible via browser

---

#### Application Web UIs (require SSH tunneling from local machine)

Since Spark runs inside a remote VM cluster, the web interfaces must be forwarded to your local machine.

```bash
ssh -L 18080:10.67.22.135:8080 -L 14040:10.67.22.135:4040 <user>@gate.cloudveneto.it
```

After connecting, open in your browser:

- http://localhost:18080 → Spark Master Web UI
- http://localhost:14040 → Spark Driver Web UI


##### 8080 → Spark Master Web UI
- Cluster-level monitoring interface
- Shows:
  - active workers
  - CPU cores per worker
  - memory allocation
  - running applications

##### 4040 → Spark Driver Web UI
- Application-level execution dashboard
- Displays:
  - stages
  - DAG (execution plan)
  - tasks
  - execution timing and metrics
- Only available while a job is running
- Disappears automatically when the application finishes

## **Example of Parallelization**

In [10]:
# local data
data = [1, 2, 3, 4, 5, 6, 7, 8]

In [11]:
# Parallelize (this distributes data to workers)
rdd = sc.parallelize(data, numSlices=4)

In [12]:
#  Action 1: map transformation (lazy)
mapped = rdd.map(lambda x: x * x)

In [13]:
# Action 2: force execution (JOB 1)
print("Squared values:", mapped.collect())

Squared values: [1, 4, 9, 16, 25, 36, 49, 64]


In [14]:
#  Action 3: second job with aggregation (JOB 2)
sum_result = rdd.reduce(lambda a, b: a + b)
print("Sum:", sum_result)

Sum: 36


In [15]:
#  what is happening 
print("Original RDD:", rdd.glom().collect())
print("Squared RDD:", mapped.glom().collect())

Original RDD: [[1, 2], [3, 4], [5, 6], [7, 8]]
Squared RDD: [[1, 4], [9, 16], [25, 36], [49, 64]]
